In [ ]:
import pandas as pd
import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np 
from sklearn.manifold import TSNE
from sklearn import cluster, datasets, mixture
import matplotlib
import cmasher as cmr
from scipy import stats
from scipy.stats.contingency import expected_freq
from scipy.stats import fisher_exact
import math 
AF2 = pd.read_pickle('../files/AF2_with_ENSG.pkl')
mdSAMs = pd.read_pickle('../files/dSAM_MPs_v_paper_v2_new_locs.pkl')
hTM = pd.read_pickle('../files/hTMs_uniprot_v1.pkl')
elm =  pd.read_csv('/Users/aelazar/notebooks_for_paper/elm/elm_all.tsv',delimiter='\t',header=5)

from random import randrange

In [ ]:
#https://academic.oup.com/bib/article/24/1/bbac599/6984799?login=false

In [ ]:
import glob 

In [ ]:
files = glob.glob('/Users/aelazar/notebooks_for_paper/conservations_UniProt/kibby/sample_data/csvs/*.csv')
accs = [i.split('/')[-1].split('.')[0] for i in files]

In [ ]:
def ranges(nums):
    nums = sorted(set(nums))
    gaps = [[s, e] for s, e in zip(nums, nums[1:]) if s+1 < e]
    edges = iter(nums[:1] + sum(gaps, []) + nums[-1:])
    return list(zip(edges, edges))
def random_idr_start(acc,motif_length):
    for r  in ranges([i[0] for i in enumerate(AF2[AF2.acc==acc].pLDDT.values[0]) if i[1]<70]):
        if r[1]-r[0]>motif_length+2:
            return int(randrange(r[0],r[1]-motif_length))
        else:
            continue
def random_structre_start(acc,motif_length):
    for r  in ranges([i[0] for i in enumerate(AF2[AF2.acc==acc].pLDDT.values[0]) if i[1]>=70]):
        if r[1]-r[0]>motif_length+2:
            return int(randrange(r[0],r[1]-motif_length))
        else:
            continue

In [ ]:
def kibby_con(acc,s,e,idrs=False):
    x  = pd.read_csv('/Users/aelazar/notebooks_for_paper/conservations_UniProt/kibby/sample_data/csvs/%s.csv'%acc,names=['AA','conservation'], header=None)
    if len(x) == len(AF2[AF2.acc==acc].AAseq.values[0]):
        l = []
        AAseq_rand =(''.join(x.iloc[s:e].AA.values))
        for ind in x.iloc[s:e].index:
            WT = x.iloc[ind].AA
            idrs 
            WT_con = x.iloc[ind].conservation
            background = x[(x.AA==WT)&(x.index.isin(idrs))].conservation.mean()
            l.append(WT_con)#/background)
        return l
    else:
        return []

In [ ]:
mdSAMs = mdSAMs.drop_duplicates(subset='AAseq')

In [ ]:
l = []

for i in tqdm.tqdm(mdSAMs[mdSAMs.acc.isin(set.intersection(set(mdSAMs.acc),set(accs)))].index):
    #idrs = [i[0] for i in enumerate(AF2[AF2.acc==acc].pLDDT.values[0]) if i[1]<70]
    acc = mdSAMs.loc[i,'acc']
    s = int(mdSAMs.loc[i,'start_new'])-1
    e = int(mdSAMs.loc[i,'end_new'])
    motif_length = int(mdSAMs.loc[i,'AA_length'])
    s_r = random_idr_start(acc,motif_length)
    ss_r = random_structre_start(acc,motif_length)
    idrs = [i[0] for i in enumerate(AF2[AF2.acc==acc].pLDDT.values[0]) if i[1]<70]
    strcutre = [i[0] for i in enumerate(AF2[AF2.acc==acc].pLDDT.values[0]) if i[1]<70]
    AAseq  = mdSAMs.loc[i,'AAseq']
    if ss_r is  not  None:
        for aa in kibby_con(acc,ss_r,ss_r+motif_length,strcutre):
            l.append({'value':aa,'ele':'random_struct','acc':acc,'ind':i})
    for aa in kibby_con(acc,s_r,s_r+motif_length,idrs):
        l.append({'value':aa,'ele':'random','acc':acc,'ind':i})
    for aa in kibby_con(acc,s,s+motif_length,idrs):
        l.append({'value':aa,'ele':'mdSAM','acc':acc,'ind':i})
c = 0
for i in elm[(elm.Primary_Acc.isin(accs))].index:
    acc = elm.loc[i,'Primary_Acc']
    s = int(elm.loc[i,'Start'])
    e  = int(elm.loc[i,'End'])
    motif_length = (e-s)+1
    if AF2[AF2.acc==acc].pLDDT.values[0]!='A':
        idrs = [i[0] for i in enumerate(AF2[AF2.acc==acc].pLDDT.values[0]) if i[1]<70]
        for aa in kibby_con(acc,s,s+motif_length,idrs):
            if aa ==np.NaN:
                print(acc)
            l.append({'value':aa,'ele':'ELM','acc':acc,'ind':c})
        c = c+1

In [ ]:
for aa in kibby_con(acc,ss_r,ss_r+motif_length,strcutre):
    print(aa)

In [ ]:
df = pd.DataFrame(l)

In [ ]:
sdf.ele

In [ ]:
df.to_pickle('ESM2_conservation.pkl')

In [ ]:
z= []
for i in r.index:
    z.append({''})
    z.append(m.loc[i[0],i[1]].values[0]-r.loc[i[0],i[1]].values[0])


In [ ]:
r = df[df.ele=='random'].groupby(['acc','ind']).median()#.columns()
m = df[df.ele=='mdSAM'].groupby(['acc','ind']).median()#.columns()
e  = df[df.ele=='ELM'].groupby(['acc','ind']).median()
s  = df[df.ele=='random_struct'].groupby(['acc','ind']).median()


In [ ]:
z = []
for i in m['value']:
    z.append({'value':i,'ele':'mdSAM'})
for i in r['value']:
    z.append({'value':i,'ele':'random'})
for i in e['value']:
    z.append({'value':i,'ele':'ELM'})
for i in s['value']:
    z.append({'value':i,'ele':'strctre'})
zz = pd.DataFrame(z)

In [ ]:

fig, ax = plt.subplots(figsize=(3, 3))
sns.boxplot(x='ele',y='value',data=zz,
            boxprops=dict(alpha=1),notch=False,saturation=0.75,
            showfliers = False,capprops=dict(linewidth=0)
           ,palette=['#06948E','#C03830','#FF9B00','gray'],order=['random','mdSAM','strctre','ELM'])
plt.ylim([-.05,.9])
ax.set_xticklabels(['$IDR_{random}$','$mdSAM$','$ELM$','$Structured$'],fontsize=10,rotation=90)
plt.ylabel('$Median\;motif\;conservarion$\n$(embedding)$')
plt.xlabel('')
plt.grid()

e = '%1.10e' %stats.ks_2samp(zz[zz.ele=='random'].value,
               zz[zz.ele=='mdSAM'].value)[-1]
lead = e[0]
power = e.split('-')[-1]#.split('0')[-1]

plt.text(1.2,.5,'$%s⋅10^{‑%s}$'%(lead,power),rotation=0,fontsize=10,color='k')

e = '%1.10e' %stats.ks_2samp(zz[zz.ele=='ELM'].value,
               zz[zz.ele=='mdSAM'].value)[-1]
lead = e[0]
power = e.split('-')[-1].split('0')[-1]

plt.text(.2,.7,'$%s⋅10^{‑%s}$'%(lead,power),rotation=0,fontsize=10,color='k')

# plt.savefig('conservarion_mean.svg',bbox_inches='tight',transparent=True)#ESP':set(ESP)
e = '%1.10e' %stats.ks_2samp(zz[zz.ele=='ELM'].value,
               zz[zz.ele=='random'].value)[-1]
lead = e[0]
power = e.split('-')[-1]#.split('0')[-1]

plt.text(.5,.9,'$%s⋅10^{‑%s}$'%(lead,power),rotation=0,fontsize=10,color='k')
plt.text(.0,.1,'$n=%d$'%len(zz[zz.ele=='ELM']),fontsize=6)
plt.text(.7,.1,'$n=%d$'%len(zz[zz.ele=='mdSAM']),fontsize=6)
plt.text(1.6,.1,'$n=%d$'%len(zz[zz.ele=='random']),fontsize=6)
plt.savefig('mean_embedding_con.svg',bbox_inches='tight',transparent=True)#ESP':set(ESP)

In [ ]:
from scipy.stats import f_oneway


In [ ]:
F, p = f_oneway(zz[zz.ele=='ELM'].value.to_numpy(),zz[zz.ele=='mdSAM'].value.to_numpy(),
                zz[zz.ele=='random'].value.to_numpy(),zz[zz.ele=='strctre'].value.to_numpy())
print(F,p)

In [ ]:
fig, ax = plt.subplots(figsize=(2.5, 2.5*1.618))
sns.boxplot(x='ele',y='value',data=df,
            boxprops=dict(alpha=1),notch=False,saturation=0.75,
            showfliers = False,capprops=dict(linewidth=0)
           ,palette=['#06948E','#C03830','#FF9B00'],order=['ELM','mdSAM','random'])
plt.ylim([-.1,1])
ax.set_xticklabels(['$ELM$','$mdSAM$','$IDR_{random}$'],fontsize=10)
plt.ylabel('$Conservarion\,(embedding)$')
plt.xlabel('')
plt.grid()
plt.savefig('venn_snp_hTM_all.svg',bbox_inches='tight',transparent=True)#ESP':set(ESP)


In [ ]:
stats.ttest_ind(df[df.ele=='random'].value,
df[df.ele=='mdSAM'].value)



In [ ]:
sns.ecdfplot(hue='ele',x='value',data=df)

In [ ]:
df

In [ ]:
sns.boxplot(x='ele',y='value',data=zz,order=[ 'random','mdSAM','strctre','ELM'])

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4*1.618))
sns.boxplot(x='ele',y='value',data=df,
            boxprops=dict(alpha=1),notch=False,saturation=0.75,
            showfliers = False,capprops=dict(linewidth=0),)

In [ ]:
zz

In [ ]:
import numpy as np
import scipy.stats as stats
from itertools import combinations

# Simulated data for four categories
categories = ['ELM', 'mdSAM', 'random', 'strctre']
data = {
    'ELM': zz[zz.ele=='ELM'].value.values,
    'mdSAM': zz[zz.ele=='mdSAM'].value.values,
    'random': zz[zz.ele=='random'].value.values,
    'strctre':zz[zz.ele=='strctre'].value.values
}

alpha = 0.05 / (len(categories) * (len(categories) - 1) / 2)  # Bonferroni-corrected alpha level

p_values = {}

for cat1, cat2 in combinations(categories, 2):
    t_stat, p_value = stats.ttest_ind(data[cat1], data[cat2])
    if cat1 not in p_values:
        p_values[cat1] = {}
    p_values[cat1][cat2] = p_value

print("P-values between pairs:")
for cat1, cat2_pvalues in p_values.items():
    for cat2, p_value in cat2_pvalues.items():
        print(f"{cat1} vs {cat2}: p-value = {p_value:.7E}")


In [ ]:
np.random.normal(10, 2, 50)

In [ ]:
conservation_df = pd.read_pickle('conservation_df.pkl')

In [ ]:
KL = conservation_df[conservation_df.Conservation_type=='Kullback-Leibler']
KL

In [ ]:
import numpy as np
import scipy.stats as stats
from itertools import combinations

# Simulated data for four categories
categories = ['IDR', 'Structred_burid', 'Structred_exposed', 'dSAM']
data = {
    'IDR': KL[KL.ele=='IDR'].conservation.dropna().values,
    'Structred_burid': KL[KL.ele=='Structred_burid'].conservation.dropna().values,
    'Structred_exposed': KL[KL.ele=='Structred_exposed'].conservation.dropna().values,
    'dSAM':KL[KL.ele=='dSAM'].conservation.dropna().values
}

alpha = 0.05 / (len(categories) * (len(categories) - 1) / 2)  # Bonferroni-corrected alpha level

p_values = {}

for cat1, cat2 in combinations(categories, 2):
    t_stat, p_value = stats.ttest_ind(data[cat1], data[cat2])
    if cat1 not in p_values:
        p_values[cat1] = {}
    p_values[cat1][cat2] = p_value

print("P-values between pairs:")
for cat1, cat2_pvalues in p_values.items():
    for cat2, p_value in cat2_pvalues.items():
        print(f"{cat1} vs {cat2}: p-value = {p_value:.7E}")


In [ ]:
from scipy.stats import tukey_hsd

In [ ]:

IDR = KL[KL.ele=='IDR'].conservation.dropna().values
Structred_burid = KL[KL.ele=='Structred_burid'].conservation.dropna().values
Structred_exposed = KL[KL.ele=='Structred_exposed'].conservation.dropna().values
dSAM  = KL[KL.ele=='dSAM'].conservation.dropna().values

In [ ]:
res =  tukey_hsd(IDR, Structred_burid, Structred_exposed,dSAM)

In [ ]:
print(res)